In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
csv_file_path = 'enter the path to your csv file here'
df = pd.read_csv(csv_file_path)

In [4]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2020-02-19,229.829895,230.305116,228.733999,229.005550
1,2020-02-20,227.696274,229.800799,224.883778,229.325593
2,2020-02-21,223.322296,226.949443,222.400963,226.561518
3,2020-02-24,214.710266,217.483975,213.614356,214.991510
4,2020-02-25,208.871887,217.231816,208.260906,216.514138
...,...,...,...,...,...
1253,2025-02-12,528.299988,529.190002,521.950012,522.299988
1254,2025-02-13,535.900024,536.219971,529.190002,529.979980
1255,2025-02-14,538.150024,538.840027,535.669983,536.010010
1256,2025-02-18,539.369995,540.000000,536.039978,539.729980


In [ ]:
def signal(data):
    signal = [0] * len(data)
    for i in range(2,len(data)):
        if ():
            signal[i] = 1
        elif ():
            signal[i] = 2
        else:
            signal[i] = 0
        data["signal"] = signal
        
signal(df)
df

In [ ]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

In [ ]:
# REVERT THE DATE COLUMN BACK TO THE INDEX
df.set_index('Date', inplace=True)

# DISPLAY A SLICE/WINDOW OF PRICE DATA
bar = 0
df1 = df[bar:bar+200].copy()

# PRICE CHART WITH INDICATOR BELOW
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

# PRICE CHART ONLY
fig = go.Figure(data=[go.Candlestick(x=df1.index,
                open=df1['Open'],
                high=df1['High'],
                low=df1['Low'],
                close=df1['Close'],
                increasing_line_color = 'rgba(19,156,19,0.8)',
                decreasing_line_color = 'rgba(175,07,49,0.8)',
                name = 'QQQ')])

# ENTRIES
fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")


# OVERLAY
fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1., 
                         opacity=0.8, 
                         line=dict(color='lightsalmon', width=1),
                        name='/'))

# CHART TITLE
fig.update_layout(
    annotations=[dict(text=" / ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=0.10,
                    y=1.05,
                    showarrow=False)])

# SUBPLOT TITLE (CHAIN FROM DICT IF YOU WANT TO SHOW BOTH)
fig.update_layout(
    annotations=[dict(text=" / ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.215,
                    showarrow=False)])

# ADD INDICATOR DATA TO INDICATOR SUBPLOT
fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1., 
                         line=dict(color='lightseagreen', width=1),
                         name='%K'),
                         row=2, col=1)

# ADD HISTOGRAM TO INDICATOR SUBPLOT
fig.add_trace(go.Bar(x=df1.index, 
                     y=df1., 
                     name="Histogram",
                     marker=dict(color='gray')),
                     row=2, col=1)

# ADD LEVELS LINE TO INDICATOR 
fig.add_hline(y=80, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

# UPDATE LAYOUT AND THEME
fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

# UPDATE GRID COLOR
fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

# DISPLAY DATE WITH NO WEEKEND DAYS AND/OR OVERNIGHT HOURS
fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
        dict(bounds=[21, 13.5], pattern="hour")
    ])

# SAVE THE CHART IMAGE TO YOUR FOLDER
fig.write_image('image.png', scale=2)

# GENERATE YOUR CHART
fig.show()


In [ ]:
# BACKTESTING SCRIPT

def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99)
        
        if len(self.trades) > 0:
            entry_time = self.data.index[self.trades[-1].entry_bar]
            current_time = self.data.index[-1]
            if (current_time - entry_time) > timedelta(days=1):
                self.trades[-1].close()
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, 
              hedging=True, trade_on_close=True, commission=0.0005)
stats = bt.run()
stats

In [ ]:
# PROFIT AND LOSS GRAPH

trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.write_image("PNL.png", scale=2)

fig_trades.show()

In [ ]:
# MONTECARLO SIMULATION

trades = stats['_trades']

def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='% / Trade',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.write_image("MC.png", scale=2)

mc.show()

In [ ]:
# TO SEE A LIST OF EVERY STAT METRIC YOU CAN ACCESS
print(stats.keys())

# SAVE A SELECTION OF STRATEGY STATS TO AN EXCEL FILE 

results = []

if stats is not None:
            results.append({
                'Start': stats['Start'],
                'End': stats['End'],
                '# Trades': stats['# Trades'],
                'Exposure Time [%]': stats['Exposure Time [%]'],
                'Win Rate [%]': stats['Win Rate [%]'],
                'Return [%]': stats['Return [%]'],
                'Buy & Hold Return [%]': stats['Buy & Hold Return [%]'],
                'Max. Drawdown [%]': stats['Max. Drawdown [%]'],
                'Avg. Drawdown [%]': stats['Avg. Drawdown [%]'],
                'Sharpe Ratio': stats['Sharpe Ratio'],
                'Sortino Ratio': stats['Sortino Ratio'],
                'Calmar Ratio': stats['Calmar Ratio'],
                'Best Trade [%]': stats['Best Trade [%]'],
                'Worst Trade [%]': stats['Worst Trade [%]'],
                'Avg. Trade [%]': stats['Avg. Trade [%]'],
                'Profit Factor': stats['Profit Factor'],
                'Expectancy [%]': stats['Expectancy [%]']
            })

results_df = pd.DataFrame(results)
results_df = results_df.T
results_df.iloc[3:] = results_df.iloc[3:].apply(lambda x: pd.to_numeric(x, errors='coerce').round(2))
results_df.to_csv('stats.csv', index=True, header=False)

In [ ]:
# SAVE TRADE RESULTS

trades = trades[['EntryTime', 'ExitTime', 'Duration', 'EntryPrice', 'ExitPrice', 'ReturnPct']]
trades['EntryPrice'] = trades['EntryPrice'].round(2)
trades['ExitPrice'] = trades['ExitPrice'].round(2)
trades['ReturnPct'] = (trades['ReturnPct'] * 100).round(2)
trades.to_csv('Alltrades.csv', index=False)


trades = trades.round(2)
last10 = trades.tail(10)
last10.to_csv('Last10.csv', index=False)